<a href="https://colab.research.google.com/github/aymensrihi/deep-learning-projects/blob/main/adaptive_selection_to_be_tested.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Visualization tools for Adaptive Swin-SPSD
Analyze stage selection patterns and performance trends
"""

import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

def plot_stage_distribution_over_time(stage_history, save_path='stage_distribution.png'):
    """
    Plot how stage selection distribution changes over epochs.

    Args:
        stage_history: List of lists, where each inner list contains stage selections for an epoch
        save_path: Path to save the plot
    """
    num_epochs = len(stage_history)

    # Calculate stage distribution per epoch
    stage_distributions = []
    for epoch_stages in stage_history:
        counter = Counter(epoch_stages)
        total = len(epoch_stages)
        dist = [counter.get(i, 0) / total * 100 for i in range(4)]
        stage_distributions.append(dist)

    stage_distributions = np.array(stage_distributions)

    # Create stacked bar chart
    fig, ax = plt.subplots(figsize=(12, 6))

    epochs = np.arange(1, num_epochs + 1)
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
    stage_labels = ['Stage 1', 'Stage 2', 'Stage 3', 'Stage 4']

    bottom = np.zeros(num_epochs)
    for i in range(4):
        ax.bar(epochs, stage_distributions[:, i], bottom=bottom,
               label=stage_labels[i], color=colors[i], alpha=0.8)
        bottom += stage_distributions[:, i]

    ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax.set_ylabel('Stage Selection %', fontsize=12, fontweight='bold')
    ax.set_title('Adaptive Stage Selection Distribution Over Time',
                 fontsize=14, fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 100)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Stage distribution plot saved to {save_path}")
    plt.close()


def plot_accuracy_and_phases(accuracy_history, stage_history,
                             low_threshold=0.4, improving_threshold=0.6,
                             save_path='accuracy_phases.png'):
    """
    Plot accuracy progression with phase annotations.

    Args:
        accuracy_history: List of epoch accuracies
        stage_history: List of lists containing stage selections
        low_threshold: Threshold for low accuracy phase
        improving_threshold: Threshold for improving phase
        save_path: Path to save the plot
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

    epochs = np.arange(1, len(accuracy_history) + 1)

    # Plot 1: Accuracy with phase regions
    ax1.plot(epochs, accuracy_history, marker='o', linewidth=2,
            color='#2E86AB', label='Training Accuracy')

    # Add phase regions
    ax1.axhspan(0, low_threshold, alpha=0.2, color='red',
               label='Early Stabilization Phase')
    ax1.axhspan(low_threshold, improving_threshold, alpha=0.2, color='orange',
               label='Transition Phase')
    ax1.axhspan(improving_threshold, 1.0, alpha=0.2, color='green',
               label='Late Refinement Phase')

    # Add threshold lines
    ax1.axhline(y=low_threshold, color='red', linestyle='--', alpha=0.7)
    ax1.axhline(y=improving_threshold, color='green', linestyle='--', alpha=0.7)

    ax1.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
    ax1.set_title('Training Accuracy Progression with Adaptive Phases',
                 fontsize=14, fontweight='bold')
    ax1.legend(loc='lower right')
    ax1.grid(alpha=0.3)
    ax1.set_ylim(0, 1.0)

    # Plot 2: Average selected stage per epoch
    avg_stages = [np.mean(epoch_stages) for epoch_stages in stage_history]

    ax2.plot(epochs, avg_stages, marker='s', linewidth=2,
            color='#A23B72', label='Average Stage Selection')
    ax2.fill_between(epochs, avg_stages, alpha=0.3, color='#A23B72')

    ax2.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Average Stage (0-3)', fontsize=12, fontweight='bold')
    ax2.set_title('Average Stage Selection Over Time',
                 fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(alpha=0.3)
    ax2.set_ylim(-0.5, 3.5)
    ax2.set_yticks([0, 1, 2, 3])
    ax2.set_yticklabels(['Stage 1', 'Stage 2', 'Stage 3', 'Stage 4'])

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Accuracy and phases plot saved to {save_path}")
    plt.close()


def plot_stage_heatmap(stage_history, save_path='stage_heatmap.png'):
    """
    Create a heatmap showing stage selection intensity over epochs and batches.

    Args:
        stage_history: List of lists containing stage selections
        save_path: Path to save the plot
    """
    # Create a 2D matrix: epochs x stage
    num_epochs = len(stage_history)
    stage_matrix = np.zeros((num_epochs, 4))

    for epoch_idx, epoch_stages in enumerate(stage_history):
        counter = Counter(epoch_stages)
        for stage in range(4):
            stage_matrix[epoch_idx, stage] = counter.get(stage, 0)

    fig, ax = plt.subplots(figsize=(10, 8))

    im = ax.imshow(stage_matrix.T, cmap='YlOrRd', aspect='auto')

    # Set ticks
    ax.set_xticks(np.arange(num_epochs))
    ax.set_yticks(np.arange(4))
    ax.set_xticklabels(np.arange(1, num_epochs + 1))
    ax.set_yticklabels(['Stage 1', 'Stage 2', 'Stage 3', 'Stage 4'])

    ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax.set_ylabel('Stage', fontsize=12, fontweight='bold')
    ax.set_title('Stage Selection Heatmap (Intensity = Selection Count)',
                fontsize=14, fontweight='bold')

    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Selection Count', rotation=270, labelpad=20, fontweight='bold')

    # Add text annotations
    for i in range(num_epochs):
        for j in range(4):
            text = ax.text(i, j, int(stage_matrix[i, j]),
                          ha="center", va="center", color="black", fontsize=9)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Stage heatmap saved to {save_path}")
    plt.close()


def plot_comparison_with_random(stage_history, save_path='comparison_random.png'):
    """
    Compare adaptive stage selection with random baseline.

    Args:
        stage_history: List of lists containing stage selections
        save_path: Path to save the plot
    """
    num_epochs = len(stage_history)

    # Calculate adaptive distribution per epoch
    adaptive_dists = []
    for epoch_stages in stage_history:
        counter = Counter(epoch_stages)
        total = len(epoch_stages)
        dist = [counter.get(i, 0) / total * 100 for i in range(4)]
        adaptive_dists.append(dist)

    adaptive_dists = np.array(adaptive_dists)

    # Random baseline (25% each stage)
    random_dists = np.ones((num_epochs, 4)) * 25

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    epochs = np.arange(1, num_epochs + 1)
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
    stage_labels = ['Stage 1', 'Stage 2', 'Stage 3', 'Stage 4']

    # Plot 1: Adaptive selection
    bottom = np.zeros(num_epochs)
    for i in range(4):
        ax1.bar(epochs, adaptive_dists[:, i], bottom=bottom,
               label=stage_labels[i], color=colors[i], alpha=0.8)
        bottom += adaptive_dists[:, i]

    ax1.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Stage Selection %', fontsize=12, fontweight='bold')
    ax1.set_title('Adaptive Stage Selection', fontsize=14, fontweight='bold')
    ax1.legend(loc='upper right')
    ax1.grid(axis='y', alpha=0.3)
    ax1.set_ylim(0, 100)

    # Plot 2: Random baseline
    bottom = np.zeros(num_epochs)
    for i in range(4):
        ax2.bar(epochs, random_dists[:, i], bottom=bottom,
               label=stage_labels[i], color=colors[i], alpha=0.8)
        bottom += random_dists[:, i]

    ax2.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Stage Selection %', fontsize=12, fontweight='bold')
    ax2.set_title('Random Stage Selection (Baseline)', fontsize=14, fontweight='bold')
    ax2.legend(loc='upper right')
    ax2.grid(axis='y', alpha=0.3)
    ax2.set_ylim(0, 100)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Comparison plot saved to {save_path}")
    plt.close()


def generate_all_plots(stage_history, accuracy_history=None,
                       low_threshold=0.4, improving_threshold=0.6,
                       output_dir='.'):
    """
    Generate all visualization plots.

    Args:
        stage_history: List of lists containing stage selections per epoch
        accuracy_history: List of epoch accuracies (optional)
        low_threshold: Low accuracy threshold
        improving_threshold: Improving accuracy threshold
        output_dir: Directory to save plots
    """
    import os
    os.makedirs(output_dir, exist_ok=True)

    print("\n" + "="*80)
    print("Generating Adaptive SPSD Visualizations")
    print("="*80)

    # Plot 1: Stage distribution over time
    plot_stage_distribution_over_time(
        stage_history,
        save_path=os.path.join(output_dir, 'stage_distribution.png')
    )

    # Plot 2: Stage heatmap
    plot_stage_heatmap(
        stage_history,
        save_path=os.path.join(output_dir, 'stage_heatmap.png')
    )

    # Plot 3: Comparison with random
    plot_comparison_with_random(
        stage_history,
        save_path=os.path.join(output_dir, 'comparison_random.png')
    )

    # Plot 4: Accuracy and phases (if accuracy provided)
    if accuracy_history is not None:
        plot_accuracy_and_phases(
            accuracy_history,
            stage_history,
            low_threshold,
            improving_threshold,
            save_path=os.path.join(output_dir, 'accuracy_phases.png')
        )

    print("="*80)
    print(f"✓ All plots saved to {output_dir}/")
    print("="*80 + "\n")


def print_stage_statistics(stage_history):
    """
    Print detailed statistics about stage selection.

    Args:
        stage_history: List of lists containing stage selections
    """
    print("\n" + "="*80)
    print("Stage Selection Statistics")
    print("="*80)

    # Overall distribution
    all_selections = [stage for epoch in stage_history for stage in epoch]
    counter = Counter(all_selections)
    total = len(all_selections)

    print("\nOverall Stage Distribution:")
    for stage in range(4):
        count = counter.get(stage, 0)
        pct = count / total * 100
        bar = "█" * int(pct / 2)
        print(f"  Stage {stage+1}: {pct:5.1f}% {bar}")

    # Epoch-by-epoch breakdown
    print("\nEpoch-by-Epoch Breakdown:")
    print("  Epoch | Stage 1 | Stage 2 | Stage 3 | Stage 4 | Avg Stage")
    print("  " + "-"*60)

    for epoch_idx, epoch_stages in enumerate(stage_history, 1):
        counter = Counter(epoch_stages)
        total_epoch = len(epoch_stages)
        dists = [counter.get(i, 0) / total_epoch * 100 for i in range(4)]
        avg_stage = np.mean(epoch_stages) + 1  # +1 for 1-indexed display

        print(f"  {epoch_idx:5d} | {dists[0]:6.1f}% | {dists[1]:6.1f}% | "
              f"{dists[2]:6.1f}% | {dists[3]:6.1f}% | {avg_stage:5.2f}")

    # Transition analysis
    print("\nPhase Transitions:")
    for epoch_idx in range(len(stage_history) - 1):
        avg_curr = np.mean(stage_history[epoch_idx])
        avg_next = np.mean(stage_history[epoch_idx + 1])
        change = avg_next - avg_curr

        if abs(change) > 0.3:
            direction = "↑" if change > 0 else "↓"
            print(f"  Epoch {epoch_idx+1}→{epoch_idx+2}: "
                  f"{direction} {abs(change):.2f} (shift to "
                  f"{'later' if change > 0 else 'earlier'} stages)")

    print("="*80 + "\n")


# Example usage script
if __name__ == "__main__":
    print("="*80)
    print("Adaptive SPSD Visualization Tool")
    print("="*80)
    print("\nThis script provides visualization functions for analyzing")
    print("adaptive stage selection patterns in Swin-SPSD training.")
    print("\nUsage:")
    print("  1. Import in your training script:")
    print("     from visualization import generate_all_plots, print_stage_statistics")
    print("\n  2. After training, call:")
    print("     generate_all_plots(stage_history, accuracy_history)")
    print("     print_stage_statistics(stage_history)")
    print("\n  3. Or use standalone after loading saved data:")
    print("     stage_history = [...]  # Load your data")
    print("     accuracy_history = [...]")
    print("     generate_all_plots(stage_history, accuracy_history)")
    print("="*80)

    # Example with synthetic data
    print("\n📊 Generating example plots with synthetic data...")

    # Create synthetic stage history showing adaptive behavior
    np.random.seed(42)
    num_epochs = 10
    stage_history_example = []

    for epoch in range(num_epochs):
        num_batches = 100

        # Simulate adaptive selection
        if epoch < 3:  # Early epochs: focus on stages 0-1
            stages = np.random.choice([0, 1], size=num_batches, p=[0.6, 0.4])
        elif epoch < 6:  # Middle epochs: transition
            stages = np.random.choice([1, 2], size=num_batches, p=[0.4, 0.6])
        else:  # Late epochs: focus on stages 2-3
            stages = np.random.choice([2, 3], size=num_batches, p=[0.4, 0.6])

        stage_history_example.append(stages.tolist())

    # Synthetic accuracy
    accuracy_history_example = [0.3 + 0.05*i + np.random.normal(0, 0.02)
                                for i in range(num_epochs)]
    accuracy_history_example = [min(0.9, max(0.2, acc))
                                for acc in accuracy_history_example]

    # Generate visualizations
    generate_all_plots(
        stage_history_example,
        accuracy_history_example,
        output_dir='example_plots'
    )

    # Print statistics
    print_stage_statistics(stage_history_example)

    print("\n✓ Example visualizations created in 'example_plots/' directory")